```c
#include <string.h>

typedef struct {
    char *key;
    struct bst *left;
    struct bst *right;
} bst;

struct bst *bst_insert(bst *t, const char *key) {
    // if current t (leaf node is NULL)
    if (t == NULL) {
        bst *new = malloc(sizeof *new);
        assert(new != NULL);

        new->key = strdup(key);
        assert(new->key != NULL);

        new->left = NULL;
        new->right = NULL;
        return new;
    }

    int cmp = strcmp(key, t->key);

    if (cmp < 0) {
        t->left = bst_insert(t->left, key);
    } else if (cmp > 0) {
        t->right = bst_insert(t->right, key);
    }

    return t;           // Outputs root tree
}

bool bst_contains(bst *t, const char *key) {
    while (t != NULL) {
        int cmp = strcmp(key, t->key);
        if (cmp == 0) return true;
        t = (cmp < 0) ? t->left : t->right;
    }

    return false;
}

void bst_free(bst *t) {
    if (t == NULL) return;
    bst_free(t->left);
    bst_free(t->right);
    free(t->key);
    free(t);
}


/*
bool bst_contains(bst *t, const char *key) {
    if (t == NULL) {
        return false;
    }
    int cmp = strcmp(key, t->key);
    if (cmp < 0) {
        return bst_contains(t->left, key);
    } else if (cmp > 0) {
        return bst_contains(t->right, key);
    }
    return true;
}
*/


```

   The `strdup()` function 



---
   The fundamental difference between `strcpy` and `strdup` lies in how they 
   handle memory allocation ... `strcpy` requires you to provide a pre-existing
   destination buffer--whether it is a fixed-size array on the stack or an
   already allocated block on the heap--and it blindly copies characterss to 
   target address till it hits '\0'... In coontrast, `strdup` manages the 
   destination memory allocation entirely automatically... it internally computes
   the length of the source string, calls `malloc` behind the scenes to 
   request a perfectly sized block of heap memory (including space for the
   hidden `\0` termintor), and then copies the text into this brand-new 
   destination.

   This distinction fundamentally alters your responsibilities regarding safety
   and memory management. Because `strcpy` does no boundary checking, passing it
   a destination buffer that is too small results in a fatal buffer overflow... that
   can corrupt adjacent variables or crash your program... While `strdup`
   ... imposes strict cleanup contract because it wraps up a hidden heap 
   allocation... every string generated by `strdup` is an independent resource
   that you are fully responsible for releasing via `free()` once it is no
   longer needed, whereas a `strcpy` target only requires freeing if the
   destination buffer itself was originally allocated on the heap...


   ...
   - USE LOCAL STACK ARRAYS (with bounded sizes): For all transient, short-lived
     tasks. If you are reading a line from a file using `fgets()`, checking a
     string configuration inside an `if` statement, or formatting text to print
     out to `stdout`, keep it on the stack. Just ensure your array size matches
     the spec requirements.
   - USE `strdup(0)` / HEAP ALLOCATIONS: Only when the string MUST OUTLIVE 
     THE CURRENT FUNCTION FRAME. If you are capturing a string to insert it into
     a long-lived ADT node (like the keys in your Analysing C ... or the rule
     database keys in Eliza)... you MUST use `strdup()`. This ensures the
     structure retains its own secure copy of the text after the parent function
     exits. 

---  

---

   ... `qsort` (Quick Sort) and `bsearch` (Binary Search) are absolute...
   Because C does not have "generics" or "templates" like C++ or Java, these 
   standard library functions use a brilliant (and slightly dangerous) workaround
   involving `void *` pinters to operate on any data type.

   ...

---
CONCEPT 1: The Generic Engine (`qsort` & `bsearch`)
   Both functions live in `<stdlib.h>` and require you to describe the exact
   memory layout of your array so the engine knows how to move bytes around
   without knowing what those bytes actually represent.

```c
typedef int (*compar)(const void *, const void *);

void qsort(void *base, size_t nmemb, size_t size, compar f);
```
   - `base`: A pointer to the start of your array.
   - `nmemb`: The number of elements in the array.
   - `size`: The size of a single element (e.g., `sizeof(int)` or 
      `sizeof(struct Player)`)
   - `compar`: A pointer to a custom function you write that teaches `qsort`
     how to comapre two elements.

   `bsearch` takes the exact same arguments, but adds a `const void *key` at the
   very beginning--a pointer to the item you are looking for.
   CRITICAL RULE: You can only use `bsearch` on an array that has already been
   sorted.

---
CONCEPT 2: The Comparator Function
   Because `qsort` doesn't know if it's sorting integers, floats, or massive
   custom structs... it passes the memory address of two elements to your
   comparator as `const void *` (read-only generic pointers).

   Your comparator must return an `int`:
   - `< 0`: Element `a` should go BEFORE Element `b`
   - `0`  : Element `a`... EQUAL
   - `> 0`: ... AFTER...

   The Three Steps of Every Comparator:
   1. CAST: Convert the `const void *` to a pointer of your actual data type.
   2. DEREFERENCE: Read the actual values from those addresses.
   3. COMPARE: Return the mathematical difference.





---

```c
#include <stdlib.h>

qsort(base, nmemb, size, comparator);
bsearch(&key, base, nmemb, size, comparator);
```

int cmp(const void *a, const void *b);


---
-- `nmemb`. number of elements in array

```c
#define NELEMENTS(arr) (sizeof(arr) / sizeof(arr[0]))

int cmp_int(const void *a, const void *b) {
    int x = *(const int *)a;
    int y = *(const int *)b;

    return (x > y) - (x < y);
}

int main(int argc, char **argv) {
    int xs[] = {5, 1, 9, 3};

    qsort(xs, NELEMENTS(xs), sizeof(xs[0]), cmp_int);

    return 0;
}
```

---


```c
#include <stdio.h>
#define NELEMENTS(arr) (sizeof(arr) / sizeof(arr[0]))

int cmp_int(const void *a, const void *b) {
    int x = *(const int *)a;
    int y = *(const int *)b;

    return (x > y) - (x < y);
}

int main() {
    int xs[] = {1, 3, 5, 9};
    int key = 9;

    int *found = bsearch(key, xs, NELEMENTS(xs), sizeof(xs[0]), cmp_int);

    if (found != NULL) {
        printf("Found %d\n", *found);
    }
}
```




---

```c
#include <stdio.h>

#define NELEMENTS(arr) (sizeof(arr) / sizeof(arr[0]))

int cmp_string_ptr(const void *a, const void *b) {
    const char *sa = *(const char *)a;
    const char *sb = *(const char *)b;
    return strcmp(sa, sb);
}

int main() {
    qsort(words, NELEMENTS(words), sizeof(words[0], cmp_string_ptr));
    return 0;
}
```

---

```c
#include <stdio.h>
#include <string.h>

#define NELEMENTS(arr) (sizeof(arr) / sizeof(arr[0]))

typedef struct {
    char name[80];
    int age;
} person;

int cmp_person_age(const void *a, const void *b) {
    const person *pa = (const person *)a;
    const person *pb = (const person *)b;

    const int age_a = pa->age;
    const int age_b = pb->age;
    return (age_a > age_b) - (age_a < age_b);
}

typedef struct {
    char first_name[80];
    char last_name[80];
} name;

int cmp_person_name(const void *a, const void *b) {
    const name *na = (const name *)a;
    const name *nb = (const name *)b;
    int cmp_first = strcmp(na->first_name, nb->first_name);
    if (cmp_first != 0) {
        return cmp_first;
    }
    return strcmp(nb->last_name, nb->last_name);
}

int main() {
    qsort(people, NELEMENTS(people), sizeof people[0], cmp_person_age);
}
```